In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yangseorin/dataset/train.csv
/kaggle/input/datasets/yangseorin/dataset/val.csv


In [2]:
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate flash-linear-attention

import os, glob; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch, time, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

D = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]) + '/'
OUT = '/kaggle/working/results/'; os.makedirs(OUT, exist_ok=True)
tr = pd.read_csv(D + 'train.csv'); va = pd.read_csv(D + 'val.csv'); print(D, tr.shape, va.shape)

NAME = 'Qwen/Qwen3.5-0.8B'
tok = AutoTokenizer.from_pretrained(NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
backbone = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).model.cuda()

TARGET_MIX = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'out_proj']
backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
backbone.enable_input_require_grads()
pm = get_peft_model(backbone, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=TARGET_MIX))
pm.print_trainable_parameters()
!nvidia-smi --query-gpu=name,memory.total --format=csv

/kaggle/input/datasets/yangseorin/dataset/ (1889, 4) (396, 4)


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


trainable params: 4,915,200 || all params: 757,308,224 || trainable%: 0.6490
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
import random, math, json, re
import torch.nn as nn
from torch.nn import functional as F
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
from scipy.optimize import linear_sum_assignment

CLASSES = sorted(tr.model.unique()); cid = {c: i for i, c in enumerate(CLASSES)}
LABEL_MAP = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
             'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash',
             'glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra',
             'gpt-6-astra','kimi-k3','qwen3.8-max']
assert CLASSES == LABEL_MAP, '순서 불일치!'; print('label_map 순서 일치 ✅')

def company(m):
    for k, v in [('claude','anthropic'),('gemini','google'),('gpt','openai'),('qwen','qwen'),('glm','zai'),('kimi','moonshot')]:
        if m.startswith(k): return v
COMP = np.array([company(c) for c in CLASSES]); COMPS = sorted(set(COMP))
M2C = torch.tensor([COMPS.index(c) for c in COMP], device='cuda')
ytr = tr.model.map(cid).values; yva = va.model.map(cid).values

def hungarian(P, pids):
    pred = P.argmax(1).copy()
    for p in np.unique(pids):
        idx = np.where(pids == p)[0]; r, c = linear_sum_assignment(-np.log(P[idx] + 1e-9)); pred[idx[r]] = c
    return pred

def evaluate(P, y, pids):
    pred = P.argmax(1); cy = np.array([COMPS.index(COMP[i]) for i in y])
    cP = np.stack([P[:, COMP == c].sum(1) for c in COMPS], 1)
    return dict(model_BA=float(balanced_accuracy_score(y, pred)), accuracy=float(accuracy_score(y, pred)),
                macro_f1=float(f1_score(y, pred, average='macro')),
                company_BA=float(balanced_accuracy_score(cy, cP.argmax(1))),
                company_BA_argmax=float(balanced_accuracy_score(cy, [COMPS.index(COMP[i]) for i in pred])),
                model_BA_hungarian=float(balanced_accuracy_score(y, hungarian(P, pids))))

def save_run(run_id, P, df, metrics, extra):
    out = pd.DataFrame(P, columns=CLASSES); out.insert(0, 'id', df.id.values)
    fname = f'val_{run_id}.csv' if 'seed' in run_id else f'val_{run_id}_seed42.csv'
    out.to_csv(OUT + fname, index=False)
    json.dump({**metrics, **extra}, open(f'{OUT}{run_id}_metrics.json', 'w'), indent=2, default=float)
    print(run_id, {k: round(v, 4) if isinstance(v, float) else v for k, v in {**metrics, **extra}.items() if k != 'history'}, flush=True)

class FlatHead(nn.Module):
    def __init__(s, d): super().__init__(); s.fc = nn.Linear(d, 18)
    def forward(s, h): return F.log_softmax(s.fc(h), -1), None

class HierHead(nn.Module):
    def __init__(s, d): super().__init__(); s.comp = nn.Linear(d, len(COMPS)); s.model = nn.Linear(d, 18)
    def forward(s, h):
        lc = F.log_softmax(s.comp(h), -1); z = s.model(h); lm = torch.empty_like(z)
        for c in range(len(COMPS)):
            idx = (M2C == c).nonzero(as_tuple=True)[0]
            lm[:, idx] = F.log_softmax(z[:, idx], -1) + lc[:, c:c+1]
        return lm, lc

def smooth_nll(logp, y, ls):
    return ((1 - ls) * -logp.gather(1, y[:, None]).squeeze(1) + ls * -logp.mean(1)).mean()

def crop(t, rng):
    s = re.split(r'(?<=[.!?])\s+', t.strip())
    if len(s) <= 2: return t
    k = rng.randint(math.ceil(0.5 * len(s)), len(s) + 1); st = rng.randint(0, len(s) - k + 1)
    return ' '.join(s[st:st + k])

def encode(texts):
    return tok(texts, padding=True, truncation=True, max_length=384, pad_to_multiple_of=64, return_tensors='pt').to('cuda')

def pool(enc):
    h = pm(**enc).last_hidden_state.float(); m = enc.attention_mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)

def feat(H, center): return torch.cat([H, H - H.mean(0, keepdim=True)], -1) if center else H

@torch.no_grad()
def predict(df, head, center):
    pm.eval(); head.eval(); out = {}
    for _, g in df.groupby('prompt_id'):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(list(g.text)))
        P = head(feat(H, center))[0].exp().cpu().numpy()
        for i, p in zip(g.index, P): out[i] = p
    return np.stack([out[i] for i in df.index])

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_run(run_id, center=True, epochs=3, lr_lora=2e-4, lr_head=1e-3, seed=42,
              ls=0.0, lora_drop=0.05, wd=0.01, crop_aug=False, hier=False, lam=0.5, rdrop=0.0):
    set_seed(seed); torch.cuda.reset_peak_memory_stats()
    for mod in pm.modules():
        if hasattr(mod, 'reset_lora_parameters'): mod.reset_lora_parameters('default', True)
        if hasattr(mod, 'lora_dropout') and 'default' in mod.lora_dropout and isinstance(mod.lora_dropout['default'], nn.Dropout):
            mod.lora_dropout['default'].p = lora_drop
    d = 1024 * (2 if center else 1); head = (HierHead(d) if hier else FlatHead(d)).cuda()
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': params, 'lr': lr_lora}, {'params': head.parameters(), 'lr': lr_head}], weight_decay=wd)
    pids = tr.prompt_id.unique(); total = epochs * len(pids); warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))
    scaler = torch.amp.GradScaler(); t0 = time.time(); hist = []

    def step_loss(texts, y):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(texts))
        lm, lc = head(feat(H, center))
        return lm, smooth_nll(lm, y, ls) + (lam * smooth_nll(lc, M2C[y], ls) if hier else 0)

    for ep in range(epochs):
        pm.train(); head.train(); rng = np.random.RandomState(seed + ep); losses = []
        for pid in rng.permutation(pids):
            g = tr[tr.prompt_id == pid]; y = torch.tensor(g.model.map(cid).values, device='cuda')
            texts = [crop(t, rng) for t in g.text] if crop_aug else list(g.text)
            lm1, loss = step_loss(texts, y)
            if rdrop > 0:
                lm2, loss2 = step_loss(texts, y)
                kl = 0.5 * (F.kl_div(lm1, lm2, log_target=True, reduction='batchmean') +
                            F.kl_div(lm2, lm1, log_target=True, reduction='batchmean'))
                loss = 0.5 * (loss + loss2) + rdrop * kl
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params + list(head.parameters()), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        Pva = predict(va, head, center); m = evaluate(Pva, yva, va.prompt_id.values)
        hist.append({'epoch': ep + 1, 'loss': float(np.mean(losses)), **m})
        print(f'[{run_id}] epoch {ep + 1} | loss {np.mean(losses):.3f} | val BA {m["model_BA"]:.4f} | +hung {m["model_BA_hungarian"]:.4f} | {time.time() - t0:.0f}s', flush=True)
    pm.save_pretrained(f'{OUT}{run_id}_lora_ep{epochs}'); torch.save(head.state_dict(), f'{OUT}{run_id}_head_ep{epochs}.pt')
    Ptr = predict(tr, head, center)
    save_run(run_id, Pva, va, m, dict(train_BA=float(balanced_accuracy_score(ytr, Ptr.argmax(1))), sec=round(time.time() - t0), seed=seed,
             peak_GB=round(torch.cuda.max_memory_allocated() / 1e9, 1), center=center, epochs=epochs, lr_lora=lr_lora, lr_head=lr_head,
             ls=ls, lora_drop=lora_drop, wd=wd, crop_aug=crop_aug, hier=hier, lam=lam, rdrop=rdrop, target=TARGET_MIX, r=16, history=hist))
    return head

# ===== V3 seed 43, 44 =====
for s in [43, 44]:
    train_run(f'V3_rdrop_seed{s}', seed=s, rdrop=1.0)
print('V3 seed 43·44 완료 ✅')

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


label_map 순서 일치 ✅


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
/tmp/ipykernel_23/888701028.py:117: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())


[V3_rdrop_seed43] epoch 1 | loss 2.149 | val BA 0.3990 | +hung 0.5909 | 1613s
[V3_rdrop_seed43] epoch 2 | loss 0.890 | val BA 0.4394 | +hung 0.6162 | 2870s
[V3_rdrop_seed43] epoch 3 | loss 0.209 | val BA 0.4798 | +hung 0.5808 | 4128s
V3_rdrop_seed43 {'model_BA': 0.4798, 'accuracy': 0.4798, 'macro_f1': 0.4763, 'company_BA': 0.6807, 'company_BA_argmax': 0.6818, 'model_BA_hungarian': 0.5808, 'train_BA': 0.9852, 'sec': 4302, 'seed': 43, 'peak_GB': 5.5, 'center': True, 'epochs': 3, 'lr_lora': 0.0002, 'lr_head': 0.001, 'ls': 0.0, 'lora_drop': 0.05, 'wd': 0.01, 'crop_aug': False, 'hier': False, 'lam': 0.5, 'rdrop': 1.0, 'target': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'out_proj'], 'r': 16}


/tmp/ipykernel_23/888701028.py:117: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())


[V3_rdrop_seed44] epoch 1 | loss 2.162 | val BA 0.3788 | +hung 0.5581 | 1258s
[V3_rdrop_seed44] epoch 2 | loss 0.929 | val BA 0.4899 | +hung 0.6162 | 2517s
[V3_rdrop_seed44] epoch 3 | loss 0.201 | val BA 0.4823 | +hung 0.6237 | 3775s
V3_rdrop_seed44 {'model_BA': 0.4823, 'accuracy': 0.4823, 'macro_f1': 0.4748, 'company_BA': 0.6856, 'company_BA_argmax': 0.6864, 'model_BA_hungarian': 0.6237, 'train_BA': 0.9873, 'sec': 3949, 'seed': 44, 'peak_GB': 5.4, 'center': True, 'epochs': 3, 'lr_lora': 0.0002, 'lr_head': 0.001, 'ls': 0.0, 'lora_drop': 0.05, 'wd': 0.01, 'crop_aug': False, 'hier': False, 'lam': 0.5, 'rdrop': 1.0, 'target': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'out_proj'], 'r': 16}
V3 seed 43·44 완료 ✅


In [4]:
import random, math, json, re
import torch.nn as nn
from torch.nn import functional as F
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
from scipy.optimize import linear_sum_assignment

CLASSES = sorted(tr.model.unique()); cid = {c: i for i, c in enumerate(CLASSES)}
LABEL_MAP = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
             'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash',
             'glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra',
             'gpt-6-astra','kimi-k3','qwen3.8-max']
assert CLASSES == LABEL_MAP, '순서 불일치!'; print('label_map 순서 일치 ✅')

def company(m):
    for k, v in [('claude','anthropic'),('gemini','google'),('gpt','openai'),('qwen','qwen'),('glm','zai'),('kimi','moonshot')]:
        if m.startswith(k): return v
COMP = np.array([company(c) for c in CLASSES]); COMPS = sorted(set(COMP))
M2C = torch.tensor([COMPS.index(c) for c in COMP], device='cuda')
ytr = tr.model.map(cid).values; yva = va.model.map(cid).values

def hungarian(P, pids):
    pred = P.argmax(1).copy()
    for p in np.unique(pids):
        idx = np.where(pids == p)[0]; r, c = linear_sum_assignment(-np.log(P[idx] + 1e-9)); pred[idx[r]] = c
    return pred

def evaluate(P, y, pids):
    pred = P.argmax(1); cy = np.array([COMPS.index(COMP[i]) for i in y])
    cP = np.stack([P[:, COMP == c].sum(1) for c in COMPS], 1)
    return dict(model_BA=float(balanced_accuracy_score(y, pred)), accuracy=float(accuracy_score(y, pred)),
                macro_f1=float(f1_score(y, pred, average='macro')),
                company_BA=float(balanced_accuracy_score(cy, cP.argmax(1))),
                company_BA_argmax=float(balanced_accuracy_score(cy, [COMPS.index(COMP[i]) for i in pred])),
                model_BA_hungarian=float(balanced_accuracy_score(y, hungarian(P, pids))))

def save_run(run_id, P, df, metrics, extra):
    out = pd.DataFrame(P, columns=CLASSES); out.insert(0, 'id', df.id.values)
    fname = f'val_{run_id}.csv' if 'seed' in run_id else f'val_{run_id}_seed42.csv'
    out.to_csv(OUT + fname, index=False)
    json.dump({**metrics, **extra}, open(f'{OUT}{run_id}_metrics.json', 'w'), indent=2, default=float)
    print(run_id, {k: round(v, 4) if isinstance(v, float) else v for k, v in {**metrics, **extra}.items() if k != 'history'}, flush=True)

class FlatHead(nn.Module):
    def __init__(s, d): super().__init__(); s.fc = nn.Linear(d, 18)
    def forward(s, h): return F.log_softmax(s.fc(h), -1), None

class HierHead(nn.Module):
    def __init__(s, d): super().__init__(); s.comp = nn.Linear(d, len(COMPS)); s.model = nn.Linear(d, 18)
    def forward(s, h):
        lc = F.log_softmax(s.comp(h), -1); z = s.model(h); lm = torch.empty_like(z)
        for c in range(len(COMPS)):
            idx = (M2C == c).nonzero(as_tuple=True)[0]
            lm[:, idx] = F.log_softmax(z[:, idx], -1) + lc[:, c:c+1]
        return lm, lc

def smooth_nll(logp, y, ls):
    return ((1 - ls) * -logp.gather(1, y[:, None]).squeeze(1) + ls * -logp.mean(1)).mean()

def crop(t, rng):
    s = re.split(r'(?<=[.!?])\s+', t.strip())
    if len(s) <= 2: return t
    k = rng.randint(math.ceil(0.5 * len(s)), len(s) + 1); st = rng.randint(0, len(s) - k + 1)
    return ' '.join(s[st:st + k])

def encode(texts):
    return tok(texts, padding=True, truncation=True, max_length=384, pad_to_multiple_of=64, return_tensors='pt').to('cuda')

def pool(enc):
    h = pm(**enc).last_hidden_state.float(); m = enc.attention_mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)

def feat(H, center): return torch.cat([H, H - H.mean(0, keepdim=True)], -1) if center else H

@torch.no_grad()
def predict(df, head, center):
    pm.eval(); head.eval(); out = {}
    for _, g in df.groupby('prompt_id'):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(list(g.text)))
        P = head(feat(H, center))[0].exp().cpu().numpy()
        for i, p in zip(g.index, P): out[i] = p
    return np.stack([out[i] for i in df.index])

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_run(run_id, center=True, epochs=3, lr_lora=2e-4, lr_head=1e-3, seed=42,
              ls=0.0, lora_drop=0.05, wd=0.01, crop_aug=False, hier=False, lam=0.5, rdrop=0.0):
    set_seed(seed); torch.cuda.reset_peak_memory_stats()
    for mod in pm.modules():
        if hasattr(mod, 'reset_lora_parameters'): mod.reset_lora_parameters('default', True)
        if hasattr(mod, 'lora_dropout') and 'default' in mod.lora_dropout and isinstance(mod.lora_dropout['default'], nn.Dropout):
            mod.lora_dropout['default'].p = lora_drop
    d = 1024 * (2 if center else 1); head = (HierHead(d) if hier else FlatHead(d)).cuda()
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': params, 'lr': lr_lora}, {'params': head.parameters(), 'lr': lr_head}], weight_decay=wd)
    pids = tr.prompt_id.unique(); total = epochs * len(pids); warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))
    scaler = torch.amp.GradScaler(); t0 = time.time(); hist = []

    def step_loss(texts, y):
        with torch.autocast('cuda', dtype=torch.float16): H = pool(encode(texts))
        lm, lc = head(feat(H, center))
        return lm, smooth_nll(lm, y, ls) + (lam * smooth_nll(lc, M2C[y], ls) if hier else 0)

    for ep in range(epochs):
        pm.train(); head.train(); rng = np.random.RandomState(seed + ep); losses = []
        for pid in rng.permutation(pids):
            g = tr[tr.prompt_id == pid]; y = torch.tensor(g.model.map(cid).values, device='cuda')
            texts = [crop(t, rng) for t in g.text] if crop_aug else list(g.text)
            lm1, loss = step_loss(texts, y)
            if rdrop > 0:
                lm2, loss2 = step_loss(texts, y)
                kl = 0.5 * (F.kl_div(lm1, lm2, log_target=True, reduction='batchmean') +
                            F.kl_div(lm2, lm1, log_target=True, reduction='batchmean'))
                loss = 0.5 * (loss + loss2) + rdrop * kl
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params + list(head.parameters()), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())
        Pva = predict(va, head, center); m = evaluate(Pva, yva, va.prompt_id.values)
        hist.append({'epoch': ep + 1, 'loss': float(np.mean(losses)), **m})
        print(f'[{run_id}] epoch {ep + 1} | loss {np.mean(losses):.3f} | val BA {m["model_BA"]:.4f} | +hung {m["model_BA_hungarian"]:.4f} | {time.time() - t0:.0f}s', flush=True)
    pm.save_pretrained(f'{OUT}{run_id}_lora_ep{epochs}'); torch.save(head.state_dict(), f'{OUT}{run_id}_head_ep{epochs}.pt')
    Ptr = predict(tr, head, center)
    save_run(run_id, Pva, va, m, dict(train_BA=float(balanced_accuracy_score(ytr, Ptr.argmax(1))), sec=round(time.time() - t0), seed=seed,
             peak_GB=round(torch.cuda.max_memory_allocated() / 1e9, 1), center=center, epochs=epochs, lr_lora=lr_lora, lr_head=lr_head,
             ls=ls, lora_drop=lora_drop, wd=wd, crop_aug=crop_aug, hier=hier, lam=lam, rdrop=rdrop, target=TARGET_MIX, r=16, history=hist))
    return head

# ===== Q4 (계층형 헤드 + R-Drop) seed 43, 44 =====
for s in [43, 44]:
    train_run(f'Q4_hier_seed{s}', seed=s, rdrop=1.0, hier=True)
print('Q4 seed 43·44 완료 ✅')

label_map 순서 일치 ✅


/tmp/ipykernel_23/2734512562.py:117: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())


[Q4_hier_seed43] epoch 1 | loss 2.894 | val BA 0.4066 | +hung 0.5707 | 1261s
[Q4_hier_seed43] epoch 2 | loss 1.154 | val BA 0.4697 | +hung 0.5884 | 2523s
[Q4_hier_seed43] epoch 3 | loss 0.302 | val BA 0.4924 | +hung 0.6136 | 3783s
Q4_hier_seed43 {'model_BA': 0.4924, 'accuracy': 0.4924, 'macro_f1': 0.491, 'company_BA': 0.6898, 'company_BA_argmax': 0.7042, 'model_BA_hungarian': 0.6136, 'train_BA': 0.9772, 'sec': 3957, 'seed': 43, 'peak_GB': 5.4, 'center': True, 'epochs': 3, 'lr_lora': 0.0002, 'lr_head': 0.001, 'ls': 0.0, 'lora_drop': 0.05, 'wd': 0.01, 'crop_aug': False, 'hier': True, 'lam': 0.5, 'rdrop': 1.0, 'target': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'out_proj'], 'r': 16}


/tmp/ipykernel_23/2734512562.py:117: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); losses.append(loss.item())


[Q4_hier_seed44] epoch 1 | loss 2.761 | val BA 0.4116 | +hung 0.5682 | 1261s
[Q4_hier_seed44] epoch 2 | loss 1.116 | val BA 0.5076 | +hung 0.6111 | 2521s
[Q4_hier_seed44] epoch 3 | loss 0.276 | val BA 0.5177 | +hung 0.6237 | 3782s
Q4_hier_seed44 {'model_BA': 0.5177, 'accuracy': 0.5177, 'macro_f1': 0.5083, 'company_BA': 0.6996, 'company_BA_argmax': 0.7076, 'model_BA_hungarian': 0.6237, 'train_BA': 0.9783, 'sec': 3956, 'seed': 44, 'peak_GB': 5.4, 'center': True, 'epochs': 3, 'lr_lora': 0.0002, 'lr_head': 0.001, 'ls': 0.0, 'lora_drop': 0.05, 'wd': 0.01, 'crop_aug': False, 'hier': True, 'lam': 0.5, 'rdrop': 1.0, 'target': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'out_proj'], 'r': 16}
Q4 seed 43·44 완료 ✅
